# Query askwri-db and try semantic search

Uses the AWS credentials in `.env` (see `docs/query-aurora.md`). Pick the **AskWRI (.venv)** kernel: Select Kernel > Jupyter Kernel... > AskWRI (.venv). Every call here is read-only.

1. Connect.
2. Run a few SQL queries.
3. Semantic search: embed a question with Cohere Embed v4 and rank chunks by pgvector cosine distance.
4. Compare with the full search the agent uses.

In [1]:
import json
import sys
from pathlib import Path

from botocore.config import Config
from IPython.display import Markdown

# The repo root, wherever the kernel starts: db/ and askwri/ import from there.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from askwri.models import MODELS
from db.query import CLUSTER, DATABASE, ENV_FILE, SECRET, read_env_file, run_sql, session_from_env

session = session_from_env(read_env_file(ENV_FILE))
print("Region:", session.region_name)

Region: us-west-2


## 1. Connect

`run_sql` from `db/query.py` waits while a paused cluster resumes (up to about a minute), so it goes first.

In [2]:
json.loads(run_sql(session, "SELECT version()")["formattedRecords"])

[{'version': 'PostgreSQL 18.6 on aarch64-linux, compiled by gcc-10.5.0, 64-bit'}]

In [3]:
# One Data API client and target for the rest of the notebook.
slow_start = Config(connect_timeout=90, read_timeout=90, retries={"max_attempts": 3})
data_api = session.client("rds-data", config=slow_start)
bedrock = session.client("bedrock-runtime", config=slow_start)
target = {
    "resourceArn": session.client("rds").describe_db_clusters(DBClusterIdentifier=CLUSTER)[
        "DBClusters"
    ][0]["DBClusterArn"],
    "secretArn": session.client("secretsmanager").describe_secret(SecretId=SECRET)["ARN"],
    "database": DATABASE,
}


def sql(statement: str, **params) -> list[dict]:
    """Run one statement with named :params (str or int) and return its rows as dicts."""
    parameters = [
        {"name": name, "value": {"longValue": v} if isinstance(v, int) else {"stringValue": v}}
        for name, v in params.items()
    ]
    response = data_api.execute_statement(
        **target, sql=statement, parameters=parameters, formatRecordsAs="JSON"
    )
    return json.loads(response["formattedRecords"])


def cell(value) -> str:
    return str(value).replace("|", "\\|").replace("\n", " ")


def show(rows: list[dict]) -> Markdown:
    """Render rows as a Markdown table."""
    if not rows:
        return Markdown("_no rows_")
    columns = list(rows[0])
    lines = ["| " + " | ".join(columns) + " |", "|" + " --- |" * len(columns)]
    lines += ["| " + " | ".join(cell(row[c]) for c in columns) + " |" for row in rows]
    return Markdown("\n".join(lines))

## 2. A few queries

In [4]:
show(
    sql(
        "SELECT (SELECT count(*) FROM documents) AS documents, (SELECT count(*) FROM chunks) AS chunks"
    )
)

| documents | chunks |
| --- | --- |
| 562 | 28098 |

In [5]:
show(sql("SELECT topic, count(*) AS documents FROM documents GROUP BY topic ORDER BY 2 DESC"))

| topic | documents |
| --- | --- |
| Climate | 186 |
| none | 67 |
| Energy | 60 |
| Cities | 58 |
| Forests | 56 |
| Food | 36 |
| Finance | 31 |
| Business | 28 |
| Ocean | 16 |
| Equity | 15 |
| Freshwater | 7 |
| Economics | 2 |

In [6]:
show(sql("SELECT kind, count(*) AS chunks FROM chunks GROUP BY kind ORDER BY 2 DESC"))

| kind | chunks |
| --- | --- |
| source_text | 24941 |
| table | 3157 |

In [7]:
# A named parameter: the newest documents from a given year on.
show(
    sql(
        """SELECT doc_id, title, pub_date, topic, doc_type FROM documents
        WHERE year >= :year ORDER BY pub_date DESC NULLS LAST LIMIT 5""",
        year=2026,
    )
)

| doc_id | title | pub_date | topic | doc_type |
| --- | --- | --- | --- | --- |
| new-clean-air-standards-for-europe-implications-for-cities | New Clean Air Standards for Europe: Implications for Cities | 2026-10-27 | Cities | working paper |
| marine-fisheries-national-biodiversity-plans | Marine Fisheries in Updated National Biodiversity Strategies and Action Plans | 2026-10-15 | Ocean | working paper |
| potential-impacts-of-the-african-continental-free-trade-area-on-the-welfare-and-adaptive-capacities-of-smallholder-farmers-in-east-africa | Potential Impacts of the African Continental Free Trade Area on the Welfare and Adaptive Capacities of Smallholder Farmers in East Africa | 2026-10-14 | Food | working paper |
| applying-integrated-urban-water-resilience-approach-six-african-cities | Applying an Integrated Urban Water Resilience Approach in Six African Cities | 2026-10-13 | Cities | research report |
| strengthening-the-role-of-blue-foods-in-national-climate-planning-and-action | Strengthening the Role of Blue Foods in National Climate Planning and Action | 2026-10-08 | Ocean | other |

## 3. Semantic search

The question is embedded the way the index was built: Cohere Embed v4, 1,024 dimensions. Questions use `input_type: search_query`; the chunks were embedded with `search_document`.

`<=>` is pgvector's cosine distance, and the HNSW index on `chunks.embedding` serves `ORDER BY ... LIMIT`. Similarity is `1 - distance`. This ranks by meaning only.

In [8]:
def embed_question(question: str) -> list[float]:
    """Cohere Embed v4 query embedding: the same model and size as chunks.embedding."""
    body = {
        "texts": [question],
        "input_type": "search_query",
        "embedding_types": ["float"],
        "output_dimension": 1024,
    }
    response = bedrock.invoke_model(modelId=MODELS["embedder"], body=json.dumps(body))
    return json.loads(response["body"].read())["embeddings"]["float"][0]


def semantic_search(question: str, k: int = 5) -> list[dict]:
    """The k chunks closest in meaning to the question, nearest first."""
    return sql(
        """SELECT round((1 - (c.embedding <=> CAST(:vec AS vector)))::numeric, 3)::float AS similarity,
          d.title, c.printed_page AS page, c.kind, left(c.content, 200) AS passage
        FROM chunks c JOIN documents d USING (doc_id)
        ORDER BY c.embedding <=> CAST(:vec AS vector)
        LIMIT :k""",
        vec=json.dumps(embed_question(question)),
        k=k,
    )

In [9]:
show(semantic_search("How fast are tropical forests being lost?"))

| similarity | title | page | kind | passage |
| --- | --- | --- | --- | --- |
| 0.57 | Hot Science | 3 | source_text | carbon loss in tropical forests increased from 0.97 petagrams of carbon (PgC) per year in 2001–2005 to 1.99 PgC per year in 2015–2019. This finding differs from the analysis of three bookkeeping model |
| 0.518 | People, Planet, Justice: Understanding and Countering Nature Crime | 12 | source_text | Tropical rainforests, critical for preventing both biodiversity loss and climate change, have been converted and destroyed across the globe, and they continue to be destroyed despite decades of politi |
| 0.497 | State of Climate Action 2025 | 48 | source_text | Keeping the Paris Agreement’s temperature limit within reach, then, will require immediate and sharp declines in permanent forest loss (Box 3)—to 1.9 million hectares per year (Mha/yr) by 2030, 1.5 Mh |
| 0.497 | Hot Science (August 2025 edition) | 2 | source_text | Warming from tropical deforestation is associated with at least 28,000 heat-related deaths each year: Forest plants help regulate local temperatures by moving water up from the soil into their roots a |
| 0.479 | State of Climate Action 2025 | 48 | source_text | Tree cover loss can occur from both natural or anthropogenic causes, including fires, logging, storms, or the conversion of forests to other land uses. In 2024, specifically, hot and dry conditions (W |

In [10]:
show(semantic_search("What drives water stress in India?"))

| similarity | title | page | kind | passage |
| --- | --- | --- | --- | --- |
| 0.515 | Climate resilient cities: Assessing differential vulnerability to climate hazards in urban India | 120 | source_text | Drought Hydrological drought results from prolonged meteorological drought, which depletes surface and sub-surface water resources (NIDM 2010). It is discussed as “water stress” at both the watershed  |
| 0.417 | Climate resilient cities: Assessing differential vulnerability to climate hazards in urban India | 121 | source_text | Many Indian cities are greatly dependent on groundwater resources for both drinking water and other needs. Central Ground Water Board (CGWB) assessments are available for multiple years and can be use |
| 0.41 | Water-Resilient Economic Development Planning in Ethiopia | 16 | source_text | Greater water stress means that Ethiopia will be more vulnerable to dry shocks. Ethiopia’s rainfall patterns are naturally variable, including distinct wet and dry seasons as well as variable amounts  |
| 0.409 | Water-Resilient Economic Development Planning in Ethiopia | 29 | table | Table C-5 \| Socioeconomic water stress exposure indicator methods  \|INDICATOR\|DATA SOURCE\|METHOD\| \|---\|---\|---\| \|Irrigation jobs exposed to high and extremely high water stress\|FDRE 2021c\|Percentage o |
| 0.399 | A Green Economy Model for India: Technical Summary of Methods and Data Used | 18 | source_text | Emissions from wastewater: Emissions from residential and commercial wastewater are calculated based on the wastewater emission intensity per capita (calculated by dividing emissions from residential  |

In [11]:
# Cohere Embed v4 is multilingual: a Spanish question still finds the English passages.
show(semantic_search("¿Cómo afecta el cambio climático a la agricultura?", k=3))

| similarity | title | page | kind | passage |
| --- | --- | --- | --- | --- |
| 0.448 | Tracking Progress Toward the Coolfood Pledge | 20 | source_text | Agriculture is responsible for 70 percent of global water withdrawals (FAO 2024a), and one-quarter of the world’s crops are grown in areas where the water supply is already highly stressed, highly unr |
| 0.442 | Methodology Underpinning the State of Climate Action Series: 2024 Update | 28 | source_text | Transforming the world’s food and agriculture sector would significantly mitigate climate change. Measures that sustainably intensify production—those that increase yields without expanding croplands  |
| 0.44 | State of Climate Action 2025 | 55 | source_text | Feeding the world’s growing population more nutritiously, equitably, and sustainably will require a combination of supply- and demand-side shifts to sustainably increase agricultural productivity, red |

## 4. Compare with the full search

`askwri.search.search_passages` is what the agent calls. It adds keyword search, fuses the two rankings, then reranks with Amazon Rerank 1.0, so `score` here is the rerank relevance. It uses the default AWS credential chain, so this cell copies the `.env` values into the kernel's environment first.

The reranker drops any hit scoring under `MIN_RELEVANCE` (0.05 in `askwri/search.py`). When this was written, the forest-loss question from section 3 came back empty: its best rerank score was 0.018, although semantic search found relevant passages. The README's example question passes the cutoff.

In [12]:
import os

os.environ.update(read_env_file(ENV_FILE))
from IPython.display import display

from askwri.search import search_passages

for question in [
    "Which countries had the highest deforestation?",
    "How fast are tropical forests being lost?",
]:
    display(Markdown(f"**{question}**"))
    hits = search_passages(question)
    display(
        show(
            [
                {
                    "score": round(hit["score"], 3),
                    "title": hit["title"],
                    "page": hit["printed_page"],
                    "kind": hit["kind"],
                    "passage": hit["content"][:200],
                }
                for hit in hits
            ]
        )
    )

**Which countries had the highest deforestation?**

| score | title | page | kind | passage |
| --- | --- | --- | --- | --- |
| 0.848 | State of Climate Action 2025 | 49 | table | TABLE B4-1 \| Trends for the top 10 countries with the highest total levels of deforestation between 2015 and 2024  \|TREE COvER EXTENT IN 2000 (MHA)\|TOTAL DEFORESTATION, 2015–24 (MHA)\|SHARE OF TOTAL GL |
| 0.728 | State of Climate Action 2025 | 49 | source_text | The top 10 countries with the highest total levels of deforestation between 2015 and 2024 collectively accounted for nearly two-thirds of all deforestation globally during this period and, therefore,  |
| 0.172 | State of Climate Action 2025 | 50 | source_text | FIGURE B4-1 \| Annual trends for the top 10 countries with the highest total levels of deforestation between 2015 and 2024  Note: Mha/yr = million hectares per year. Sources: Hansen et al. 2013; Turuba |

**How fast are tropical forests being lost?**

_no rows_